In [1]:
from pathlib import Path

import pandas as pd

# Carpeta con los CSV generados por `transform`
CSV_DIR = Path("../data/csv")

# Fecha en la que, según recuerdo (no verificado), el límite de duración
# de un Short pasó de 60 s a 180 s
CAMBIO_LIMITE_SHORTS = pd.Timestamp("2024-10-15", tz="UTC")

videos = pd.read_csv(CSV_DIR / "videos.csv")
videos["published_at"] = pd.to_datetime(videos["published_at"], utc=True)
videos["is_public"] = videos["privacy_status"] == "public"

# --- 4.1 Recuento por formato ----------------------------------------------
print("== 4.1 Vídeos por formato ==")
print(videos["format"].value_counts(dropna=False).to_string())

# --- 4.2 Formato según el método que lo decidió ----------------------------
print("\n== 4.2 Formato x método de clasificación (todos) ==")
print(pd.crosstab(videos["format"], videos["format_source"], margins=True).to_string())

# --- 4.3 Mismo cruce separando públicos y no públicos ----------------------
print("\n== 4.3 Públicos (True) vs no públicos (False) ==")
print(
    pd.crosstab(
        [videos["is_public"], videos["format"]],
        videos["format_source"],
        margins=True,
    ).to_string()
)

# --- 4.4 Incoherencias entre formato y duración ----------------------------
# Límite de duración aplicable a cada vídeo según su fecha de publicación
videos["limite_short_s"] = videos["published_at"].ge(CAMBIO_LIMITE_SHORTS).map(
    {True: 180, False: 60}
)

# Short con duración superior al máximo permitido: casi seguro mal clasificado
short_demasiado_largo = (videos["format"] == "short") & (
    videos["duration_seconds"] > videos["limite_short_s"]
)
# Long con duración dentro del límite de un Short: ambiguo (puede ser un vídeo
# corto horizontal, o un Short que no se detectó)
long_dentro_del_limite = (videos["format"] == "long") & (
    videos["duration_seconds"] <= videos["limite_short_s"]
)

print("\n== 4.4 Incoherencias formato/duración ==")
resumen = pd.DataFrame(
    {
        "short_demasiado_largo": short_demasiado_largo.groupby(videos["is_public"]).sum(),
        "long_dentro_del_limite": long_dentro_del_limite.groupby(videos["is_public"]).sum(),
    }
)
resumen.index.name = "is_public"
print(resumen.to_string())

columnas = [
    "video_id", "title", "privacy_status", "published_at",
    "format", "format_source", "duration_seconds", "view_count",
]
publicos = videos["is_public"]

print("\n-- Públicos: short con duración superior al límite --")
print(videos.loc[short_demasiado_largo & publicos, columnas].to_string(index=False))

print("\n-- Públicos: long con duración dentro del límite de un Short --")
print(videos.loc[long_dentro_del_limite & publicos, columnas].to_string(index=False))

# --- 4.5 Distribución de duración por formato (solo públicos) --------------
print("\n== 4.5 Duración (s) por formato, solo públicos ==")
print(
    videos.loc[publicos]
    .groupby("format")["duration_seconds"]
    .describe()
    .round(0)
    .to_string()
)

== 4.1 Vídeos por formato ==
format
long     272
short     99
live       4

== 4.2 Formato x método de clasificación (todos) ==
format_source  duration_heuristic_nonpublic  live_streaming_details  shorts_url_probe  All
format                                                                                    
live                                      0                       4                 0    4
long                                     10                       0               262  272
short                                     7                       0                92   99
All                                      17                       4               354  375

== 4.3 Públicos (True) vs no públicos (False) ==
format_source     duration_heuristic_nonpublic  live_streaming_details  shorts_url_probe  All
is_public format                                                                             
False     live                               0                       1                 0